In [1]:
import numpy as np
import pandas as pd

In [18]:
def convert_positions_parquet(parquet_file,
                              output_dir=None,
                              output_prefix=None,
                              x_col="x",
                              y_col="y",
                              z_col="z",
                              write_tsv=True,
                              write_vts=True,
                              write_vrt=True):
    """
    Converte um arquivo .parquet de posições para .tsv, .vts e .vrt.

    Mantém as mesmas colunas do DataFrame original.
    Para .vts/.vrt:
        - usa x, y, z como coordenadas espaciais dos pontos;
        - salva todas as colunas originais como PointData.

    Observação:
        .vts é um formato VTK XML StructuredGrid.
        .vrt não é uma extensão VTK padrão para StructuredGrid, mas aqui é salvo
        com o mesmo conteúdo XML do .vts, caso você queira manter essa extensão.

    Args:
        parquet_file: caminho do arquivo .parquet.
        output_dir: pasta de saída. Se None, usa a pasta do parquet.
        output_prefix: prefixo dos arquivos. Se None, usa o nome do parquet.
        x_col, y_col, z_col: nomes das colunas de coordenadas.
        write_tsv, write_vts, write_vrt: controla quais formatos salvar.

    Returns:
        dict com os caminhos salvos.
    """
    import os
    import numpy as np
    import pandas as pd
    from xml.sax.saxutils import escape

    parquet_file = os.path.abspath(parquet_file)

    if not os.path.exists(parquet_file):
        raise FileNotFoundError(f"Arquivo não encontrado: {parquet_file}")

    df = pd.read_parquet(parquet_file)

    required = [x_col, y_col, z_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"O parquet precisa conter as colunas {required}. "
            f"Colunas ausentes: {missing}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

    if df.empty:
        raise ValueError("O DataFrame está vazio. Nada para converter.")

    if output_dir is None:
        output_dir = os.path.dirname(parquet_file)

    os.makedirs(output_dir, exist_ok=True)

    if output_prefix is None:
        output_prefix = os.path.splitext(os.path.basename(parquet_file))[0]

    out_paths = {}

    # ------------------------------------------------------------
    # 1) TSV: mantém exatamente a estrutura tabular
    # ------------------------------------------------------------
    if write_tsv:
        tsv_path = os.path.join(output_dir, f"{output_prefix}.tsv")
        df.to_csv(tsv_path, sep="\t", index=False)
        out_paths["tsv"] = tsv_path

    # ------------------------------------------------------------
    # Helpers para escrita VTK XML ASCII
    # ------------------------------------------------------------
    def _vtk_type(dtype):
        dtype = np.dtype(dtype)

        if np.issubdtype(dtype, np.floating):
            if dtype.itemsize <= 4:
                return "Float32"
            return "Float64"

        if np.issubdtype(dtype, np.signedinteger):
            if dtype.itemsize <= 4:
                return "Int32"
            return "Int64"

        if np.issubdtype(dtype, np.unsignedinteger):
            if dtype.itemsize <= 4:
                return "UInt32"
            return "UInt64"

        raise TypeError(f"Tipo não suportado para VTK XML: {dtype}")

    def _array_to_ascii(values):
        arr = np.asarray(values)

        if np.issubdtype(arr.dtype, np.floating):
            return " ".join(f"{v:.17g}" for v in arr)

        return " ".join(str(int(v)) for v in arr)

    def _write_structured_grid_xml(path):
        n_points = len(df)
        extent = f"0 {n_points - 1} 0 0 0 0"

        numeric_cols = []
        skipped_cols = []

        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_cols.append(col)
            else:
                skipped_cols.append(col)

        if skipped_cols:
            print(
                "[WARN] As seguintes colunas não são numéricas e não foram "
                f"salvas no VTK XML: {skipped_cols}"
            )

        x = df[x_col].to_numpy(dtype=np.float64)
        y = df[y_col].to_numpy(dtype=np.float64)
        z = df[z_col].to_numpy(dtype=np.float64)

        points = np.column_stack([x, y, z])

        with open(path, "w", encoding="utf-8") as f:
            f.write('<?xml version="1.0"?>\n')
            f.write(
                '<VTKFile type="StructuredGrid" version="0.1" '
                'byte_order="LittleEndian">\n'
            )
            f.write(f'  <StructuredGrid WholeExtent="{extent}">\n')
            f.write(f'    <Piece Extent="{extent}">\n')

            # PointData: mantém todas as colunas numéricas do parquet
            f.write('      <PointData>\n')
            for col in numeric_cols:
                arr = df[col].to_numpy()
                vtk_type = _vtk_type(arr.dtype)
                col_name = escape(str(col))

                f.write(
                    f'        <DataArray type="{vtk_type}" '
                    f'Name="{col_name}" format="ascii">\n'
                )
                f.write("          ")
                f.write(_array_to_ascii(arr))
                f.write("\n")
                f.write("        </DataArray>\n")

            f.write('      </PointData>\n')

            # CellData vazio
            f.write('      <CellData>\n')
            f.write('      </CellData>\n')

            # Coordenadas espaciais dos pontos
            f.write('      <Points>\n')
            f.write(
                '        <DataArray type="Float64" '
                'NumberOfComponents="3" format="ascii">\n'
            )

            f.write("          ")
            f.write(" ".join(f"{v:.17g}" for v in points.ravel()))
            f.write("\n")

            f.write("        </DataArray>\n")
            f.write("      </Points>\n")

            f.write("    </Piece>\n")
            f.write("  </StructuredGrid>\n")
            f.write("</VTKFile>\n")

    # ------------------------------------------------------------
    # 2) VTS: VTK StructuredGrid
    # ------------------------------------------------------------
    if write_vts:
        vts_path = os.path.join(output_dir, f"{output_prefix}.vts")
        _write_structured_grid_xml(vts_path)
        out_paths["vts"] = vts_path

    # ------------------------------------------------------------
    # 3) VRT: mesmo conteúdo XML do VTS, mas com extensão .vrt
    # ------------------------------------------------------------
    if write_vrt:
        vrt_path = os.path.join(output_dir, f"{output_prefix}.vrt")
        _write_structured_grid_xml(vrt_path)
        out_paths["vrt"] = vrt_path

    print("Arquivos convertidos:")
    for fmt, path in out_paths.items():
        print(f"  {fmt}: {path}")

    return out_paths

In [20]:
convert_positions_parquet(
    "../network/Positions_Carmona/3D_L512_nc1_c0.01_fT0.07_BIGGESTCOMPONENT.parquet"
)

Arquivos convertidos:
  tsv: /home/light/Documents/SOP-NetworkTests/network/Positions_Carmona/3D_L512_nc1_c0.01_fT0.07_BIGGESTCOMPONENT.tsv
  vts: /home/light/Documents/SOP-NetworkTests/network/Positions_Carmona/3D_L512_nc1_c0.01_fT0.07_BIGGESTCOMPONENT.vts
  vrt: /home/light/Documents/SOP-NetworkTests/network/Positions_Carmona/3D_L512_nc1_c0.01_fT0.07_BIGGESTCOMPONENT.vrt


{'tsv': '/home/light/Documents/SOP-NetworkTests/network/Positions_Carmona/3D_L512_nc1_c0.01_fT0.07_BIGGESTCOMPONENT.tsv',
 'vts': '/home/light/Documents/SOP-NetworkTests/network/Positions_Carmona/3D_L512_nc1_c0.01_fT0.07_BIGGESTCOMPONENT.vts',
 'vrt': '/home/light/Documents/SOP-NetworkTests/network/Positions_Carmona/3D_L512_nc1_c0.01_fT0.07_BIGGESTCOMPONENT.vrt'}